In [1]:
from IPython.display import HTML

In [2]:
import boto3
import sagemaker
from sagemaker import get_execution_role
from sagemaker.mxnet import MXNet

In [3]:
get_ipython().system('cat mnist.py')

import logging

import gzip
import mxnet as mx
import numpy as np
import os
import struct


def load_data(path):
    with gzip.open(find_file(path, "labels.gz")) as flbl:
        struct.unpack(">II", flbl.read(8))
        labels = np.fromstring(flbl.read(), dtype=np.int8)
        
    with gzip.open(find_file(path, "images.gz")) as fimg:
        _, _, rows, cols = struct.unpack(">IIII", fimg.read(16))
        images = np.fromstring(fimg.read(), dtype=np.uint8).reshape(len(labels), rows, cols)
        images = images.reshape(images.shape[0], 1, 28, 28).astype(np.float32) / 255
        
    return labels, images


def find_file(root_path, file_name):
    for root, dirs, files in os.walk(root_path):
        if file_name in files:
            return os.path.join(root, file_name)


def build_graph():
    data = mx.sym.var('data')
    data = mx.sym.flatten(data=data)
    
    fc1 = mx.sym.FullyConnected(data=data, num_hidden=128)
    act1 = mx.sym.Activation(data=fc1, act_type="relu")
    
 

In [4]:
region = boto3.Session().region_name
region

'us-east-1'

In [5]:
custom_code_upload_location = 's3://alexbc-sagemaker-us/sagemaker/customcode/mxnet'
model_artifacts_location = 's3://alexbc-sagemaker-us/sagemaker/artifacts'


In [6]:
train_data_location = 's3://sagemaker-sample-data-{}/mxnet/mnist/train'.format(region)
test_data_location = 's3://sagemaker-sample-data-{}/mxnet/mnist/test'.format(region)


In [7]:
role = get_execution_role()
role


'arn:aws:iam::771620814942:role/service-role/AmazonSageMaker-ExecutionRole-20171212T090635'

In [8]:

mnist_estimator = MXNet(entry_point='mnist.py',
                        role=role,
                        output_path=model_artifacts_location,
                        code_location=custom_code_upload_location,
                        train_instance_count=1, 
                        train_instance_type='ml.m4.xlarge',
                        hyperparameters={'learning_rate': 0.1})


In [9]:
mnist_estimator.fit({'train': train_data_location, 'test': test_data_location})

INFO:sagemaker:Creating training-job with name: sagemaker-mxnet-2018-04-28-18-48-39-663


......................................
2018-04-28 18:51:31,316 INFO - root - running container entrypoint
2018-04-28 18:51:31,316 INFO - root - starting train task
2018-04-28 18:51:31,322 INFO - container_support.training - Training starting
2018-04-28 18:51:33,452 INFO - mxnet_container.train - MXNetTrainingEnvironment: {'enable_cloudwatch_metrics': False, 'available_gpus': 0, 'channels': {u'test': {u'TrainingInputMode': u'File', u'RecordWrapperType': u'None', u'S3DistributionType': u'FullyReplicated'}, u'train': {u'TrainingInputMode': u'File', u'RecordWrapperType': u'None', u'S3DistributionType': u'FullyReplicated'}}, '_ps_verbose': 0, 'resource_config': {u'current_host': u'algo-1', u'hosts': [u'algo-1']}, 'user_script_name': u'mnist.py', 'input_config_dir': '/opt/ml/input/config', 'channel_dirs': {u'test': u'/opt/ml/input/data/test', u'train': u'/opt/ml/input/data/train'}, 'code_dir': '/opt/ml/code', 'output_data_dir': '/opt/ml/output/data/', 'output_dir': '/opt/ml/output', 'model_d

2018-04-28 18:51:47,765 INFO - root - Epoch[8] Batch [300]#011Speed: 49184.07 samples/sec#011accuracy=0.971400
2018-04-28 18:51:47,971 INFO - root - Epoch[8] Batch [400]#011Speed: 48539.01 samples/sec#011accuracy=0.972200
2018-04-28 18:51:48,199 INFO - root - Epoch[8] Batch [500]#011Speed: 43901.62 samples/sec#011accuracy=0.972600
2018-04-28 18:51:48,388 INFO - root - Epoch[8] Train-accuracy=0.975354
2018-04-28 18:51:48,388 INFO - root - Epoch[8] Time cost=1.285
2018-04-28 18:51:48,500 INFO - root - Epoch[8] Validation-accuracy=0.966000
2018-04-28 18:51:48,767 INFO - root - Epoch[9] Batch [100]#011Speed: 37838.51 samples/sec#011accuracy=0.978218
2018-04-28 18:51:48,999 INFO - root - Epoch[9] Batch [200]#011Speed: 43197.27 samples/sec#011accuracy=0.978500
2018-04-28 18:51:49,231 INFO - root - Epoch[9] Batch [300]#011Speed: 43010.56 samples/sec#011accuracy=0.974700
2018-04-28 18:51:49,465 INFO - root - Epoch[9] Batch [400]#011Speed: 42800.16 samples/sec#011accuracy=0.976300
2018-04-28 18

2018-04-28 18:52:05,236 INFO - root - Epoch[19] Validation-accuracy=0.972600
2018-04-28 18:52:05,450 INFO - root - Epoch[20] Batch [100]#011Speed: 47226.61 samples/sec#011accuracy=0.995545
2018-04-28 18:52:05,663 INFO - root - Epoch[20] Batch [200]#011Speed: 46952.55 samples/sec#011accuracy=0.995300
2018-04-28 18:52:05,905 INFO - root - Epoch[20] Batch [300]#011Speed: 41341.44 samples/sec#011accuracy=0.994700
2018-04-28 18:52:06,129 INFO - root - Epoch[20] Batch [400]#011Speed: 44633.88 samples/sec#011accuracy=0.996100
2018-04-28 18:52:06,347 INFO - root - Epoch[20] Batch [500]#011Speed: 45953.14 samples/sec#011accuracy=0.996000
2018-04-28 18:52:06,561 INFO - root - Epoch[20] Train-accuracy=0.995960
2018-04-28 18:52:06,561 INFO - root - Epoch[20] Time cost=1.325
2018-04-28 18:52:06,665 INFO - root - Epoch[20] Validation-accuracy=0.973200
2018-04-28 18:52:06,900 INFO - root - Epoch[21] Batch [100]#011Speed: 42877.95 samples/sec#011accuracy=0.995941
2018-04-28 18:52:07,142 INFO - root - 

In [10]:
predictor = mnist_estimator.deploy(initial_instance_count=1,
                                   instance_type='ml.m4.xlarge')


INFO:sagemaker:Creating model with name: sagemaker-mxnet-2018-04-28-18-48-39-663
INFO:sagemaker:Creating endpoint with name sagemaker-mxnet-2018-04-28-18-48-39-663


-------------------------------------------------------------!

In [11]:
HTML(open("input.html").read())

In [12]:
print(data)

[[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 